# STAC discovery: a footprint is not a raster measurement

What does a satellite catalog tell us before we download a single pixel? Compare a small Sentinel search with a WorldPop Degree of Urbanisation item, keeping product semantics and terms visible.

## Objectives

- Inspect Item, geometry, asset and collection metadata
- Plot footprints, acquisition times and item cloud metadata
- Separate discovery, pixel analysis and population inference

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import repo_examples as rx

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("sentinel"))
display(rx.source_card("worldpop"))

In [ ]:
search = rx.read_json("stac/sentinel-search.json")
sentinel_collection = rx.read_json("stac/sentinel-collection.json")
worldpop = rx.read_json("stac/worldpop-item.json")
worldpop_collection = rx.read_json("stac/worldpop-collection.json")
assert len(search["features"]) == 8 and worldpop["id"] == "UGA_DUG_2020"
items = pd.DataFrame([{"id": f["id"], "datetime": f["properties"]["datetime"],
                       "cloud_percent": f["properties"].get("eo:cloud_cover"), "asset_count": len(f["assets"]),
                       "geometry_type": f["geometry"]["type"]} for f in search["features"]])
items["datetime"] = pd.to_datetime(items.datetime, utc=True)
display(items)
display(pd.DataFrame([{"collection": "Sentinel-2 L2A", "license_label": sentinel_collection["license"], "meaning": "Sentinel legal notice, not CC/MIT"},
                      {"collection": "WorldPop UGA", "license_label": worldpop_collection["license"], "meaning": "DUG classes, reference year 2020; metadata only"}]))

## View 1: inspect the returned footprints

Polygons show catalog coverage, not cloud-free pixels or the ground area of a health exposure. The dashed box is the requested search region; STAC intersection search can return footprints extending beyond it. The WorldPop item has a different geographic extent and product meaning and therefore receives its own axes.

**Method connection:** the catalog's Cartesian coordinates (§2), Table (§3) and Scatterplot (§9) transfer to footprint location, acquisition ordering and metadata comparison. Geographic footprint maps are a spatial extension; a choropleth would require meaningful areal values and denominators absent here. The Gaborone TB story repository is a narrative reference, not evidence that these scenes measure TB incidence.

In [ ]:
def polygon_rings(geometry):
    if geometry["type"] == "Polygon":
        return [geometry["coordinates"][0]]
    if geometry["type"] == "MultiPolygon":
        return [polygon[0] for polygon in geometry["coordinates"]]
    raise ValueError("Expected Polygon or MultiPolygon footprint")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, feature in enumerate(search["features"]):
    for ring in polygon_rings(feature["geometry"]):
        xy = np.asarray(ring)
        axes[0].fill(xy[:, 0], xy[:, 1], alpha=.12, edgecolor="#1d6a8c")
    center = np.asarray(polygon_rings(feature["geometry"])[0]).mean(axis=0)
    axes[0].text(*center, str(i+1), fontsize=8)
axes[0].plot([25.7, 26.2, 26.2, 25.7, 25.7], [-24.9, -24.9, -24.4, -24.4, -24.9], "k--", label="Query bbox")
for ring in polygon_rings(worldpop["geometry"]):
    xy = np.asarray(ring)
    axes[1].fill(xy[:, 0], xy[:, 1], alpha=.3, color="#c88135")
for ax, title in zip(axes, ["8 returned Sentinel Items (metadata)", "UGA_DUG_2020 footprint (metadata)"]):
    rx.finish(ax, title, "Longitude (degrees east)", "Latitude (degrees north)")
    ax.set_aspect("equal")
axes[0].legend()
fig.tight_layout()
plt.show()

## View 2: cloud metadata helps choose candidates

`eo:cloud_cover` describes an item-level estimate. It does not establish cloud percentage within our small bounding box. Two overlapping Items can share a time while covering different tiles. This fixed response is the first eight returned Items, not an exhaustive January inventory or a random sample.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(items.datetime, items.cloud_percent, s=65, color="#216d8c")
for i, row in items.iterrows():
    axes[0].annotate(str(i+1), (row.datetime, row.cloud_percent), xytext=(4, 3), textcoords="offset points")
rx.finish(axes[0], "Item-level cloud metadata", "Acquisition UTC", "eo:cloud_cover (%)")
axes[0].tick_params(axis="x", labelrotation=35)
axes[1].barh(np.arange(1, 9), items.asset_count, color="#c88135")
rx.finish(axes[1], "Asset references per returned Item", "Number of asset references", "Item number in table")
fig.tight_layout()
plt.show()

## View 3: assets and legal metadata are part of discovery

An asset URL identifies a candidate resource; it does not prove anonymous browser access, CORS, HTTP Range support, a usable CRS or a permitted redistribution. We show references without fetching them. WorldPop's DUG product is a classification reference, not a population-count layer. A 2020 reference year and 2026 publication can both be correct.

In [ ]:
asset_rows = []
for label, item in [("Sentinel first returned item", search["features"][0]), ("WorldPop DUG 2020", worldpop)]:
    for key, asset in item["assets"].items():
        asset_rows.append({"item": label, "asset": key, "roles": ", ".join(asset.get("roles", [])),
                           "media_type": asset.get("type", "not supplied"), "href": asset["href"]})
display(HTML(pd.DataFrame(asset_rows).to_html(index=False, render_links=True, escape=True)))
asset_types = pd.crosstab(pd.DataFrame(asset_rows).media_type, pd.DataFrame(asset_rows).item)
fig, ax = plt.subplots(figsize=(11, 5))
asset_types.plot.barh(ax=ax, color=["#377b92", "#c88135"])
rx.finish(ax, "Media-type references: first Sentinel Item vs WorldPop Item", "Asset references (not downloaded pixels)", "Reported media type")
fig.tight_layout()
plt.show()
display({"WorldPop_properties": worldpop["properties"], "WorldPop_citation_links": [x for x in worldpop.get("links", []) if x.get("rel") in ["license", "cite-as"]]})
display({"Sentinel_collection_license_links": [x for x in sentinel_collection["links"] if x.get("rel") == "license"],
         "query": rx.manifest()["files"]["stac/sentinel-search.json"]["url"]})

## Exercise

Select a lower-cloud candidate from the table. Explain why the selection does not guarantee cloud-free coverage over Gaborone. Identify one asset's media type and the additional CRS, scale, units, QA and license checks required for pixel analysis. Sketch a bounded pixel-analysis plan without downloading any pixels.

## Limitations and reflection

Only catalog metadata is included. No raster pixels, urban class proportions, population counts or health measurements are derived. Search ordering and sample limits matter. WorldPop's item/collection CC BY evidence and Sentinel's proprietary collection label plus Sentinel legal notice are retained separately. Native GDAL/RustGIS and authenticated imagery downloads are external continuation routes.

## Citations and repository connections

[Fixed Earth Search query](https://earth-search.aws.element84.com/v1/search?collections=sentinel-2-l2a&bbox=25.7%2C-24.9%2C26.2%2C-24.4&datetime=2024-01-01T00%3A00%3A00Z%2F2024-01-31T23%3A59%3A59Z&limit=8), [WorldPop item](https://api.stac.worldpop.org/collections/UGA/items/UGA_DUG_2020), [WorldPop DOI](https://doi.org/10.5258/SOTON/WP00879), [Sentinel legal notice](https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice). Repository connections: [STAC Browser fa623cc4ef5f](https://github.com/jltobias/JupyterLite-STAC-Browser/tree/fa623cc4ef5fc472da3c0237e6d270980f2273b1), [RustGIS 64ee4e32222c](https://github.com/jltobias/JupyterLite-RustGIS/tree/64ee4e32222c07006ab28a634b759d985a002bda), [Gaborone story a89bafaa4920](https://github.com/jltobias/JupyterGIS-StoryMap-Gaborone-TB-ABM/tree/a89bafaa49202382f5931645ab0100a37179bc36).

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.